# Moduł 11. Python Unit Tests — `hypothesis` — v2.0

Autor: Kamil Bartocha

## Rozkład jazdy

- 🔹 1. Property-based testing - filozofia i zalety
- 🔹 2. `@given` i strategie (`st.integers`, `st.text`, `st.lists`)
- 🔹 3. `assume`, `composite` i strategie złożone

## 1. 🔹 Property-based testing - filozofia i zalety

Testy przykładowe (example-based) sprawdzają konkretne wartości:
`assert add(2, 3) == 5`. Testowanie własności (property-based)
formułuje ogólne prawa i sprawdza je na setkach wygenerowanych
przypadków:

```
Dla dowolnych a, b: add(a, b) == add(b, a)
Dla dowolnej listy lst: len(sorted(lst)) == len(lst)
```

Biblioteka `hypothesis` automatycznie:

1. Generuje przypadki testowe (domyślnie 100)
2. Wykrywa kontrprzykład naruszający własność
3. Minimalizuje (shrinks) kontrprzykład do najprostszej postaci

Instalacja: `pip install hypothesis`

Porównanie podejść:

| | Example-based | Property-based |
|---|---|---|
| Przypadki | ręcznie wybrane | automatycznie generowane |
| Pokrycie | zależy od testera | szersze, losowe |
| Czytelność | wysoka | wymaga myślenia o ogólnych prawach |
| Edge cases | łatwo pominąć | hypothesis szuka aktywnie |

PBT nie zastępuje testów przykładowych - uzupełnia je.

In [ ]:
import sys, subprocess, tempfile, os

def _run(code, flags=None):
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='_test.py', delete=False, dir='.'
    ) as tmp:
        tmp.write(code)
        fname = tmp.name
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', fname] + flags,
        capture_output=True, text=True
    )
    short = os.path.basename(fname)
    print(result.stdout.replace(fname, short)[-2000:])
    os.unlink(fname)

# Przykład: property test dla sorted()
_run('''
from hypothesis import given
import hypothesis.strategies as st

# Własność 1: sortowanie nie zmienia liczby elementów
@given(st.lists(st.integers()))
def test_sort_preserves_length(lst):
    assert len(sorted(lst)) == len(lst)

# Własność 2: wynik sortowania jest posortowany
@given(st.lists(st.integers()))
def test_sort_result_is_sorted(lst):
    result = sorted(lst)
    for i in range(len(result) - 1):
        assert result[i] <= result[i + 1]

# Własność 3: sortowanie listy jednoelementowej
@given(st.integers())
def test_sort_single_element(x):
    assert sorted([x]) == [x]
''')

---

### 🐍 Ćwiczenia — własności podstawowe

1. Napisz property test dla `sorted(list)` - wynik jest posortowany
2. Sprawdź własność `reverse(reverse(lst)) == lst`
8. Testuj funkcję sortowania przez własność
   `len(sorted) == len(original)`

In [ ]:
# Ćwiczenie 1: sorted(list) - wynik posortowany
# Napisz dwa property testy dla sorted():
#   a) każdy element wyniku <= następny element
#   b) zbiór elementów nie zmienia się po sortowaniu

_run('''
from hypothesis import given
import hypothesis.strategies as st

@given(st.lists(st.integers()))
def test_sorted_is_ordered(lst):
    result = sorted(lst)
    ...

@given(st.lists(st.integers()))
def test_sorted_same_elements(lst):
    ...
''')

In [ ]:
# Ćwiczenie 2: reverse(reverse(lst)) == lst
# Własność: odwrócenie listy dwa razy zwraca oryginał.
# Sprawdź dla list zawierających liczby całkowite i teksty.

_run('''
from hypothesis import given
import hypothesis.strategies as st

@given(st.lists(st.integers()))
def test_reverse_twice_integers(lst):
    ...

@given(st.lists(st.text(max_size=10)))
def test_reverse_twice_strings(lst):
    ...
''')

In [ ]:
# Ćwiczenie 8: len(sorted) == len(original)
# Napisz własną funkcję merge_sort i sprawdź że
# nie zmienia liczby elementów ani ich zbioru.

_run('''
from hypothesis import given
import hypothesis.strategies as st

def merge_sort(lst):
    if len(lst) <= 1:
        return lst
    mid = len(lst) // 2
    left = merge_sort(lst[:mid])
    right = merge_sort(lst[mid:])
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

@given(st.lists(st.integers()))
def test_merge_sort_length(lst):
    assert len(merge_sort(lst)) == ...

@given(st.lists(st.integers()))
def test_merge_sort_same_elements(lst):
    assert sorted(merge_sort(lst)) == ...
''')

## 2. 🔹 `@given` i strategie (`st.integers`, `st.text`, `st.lists`)

Strategie (`hypothesis.strategies` jako `st`) opisują jak generować
dane wejściowe. Najczęściej używane:

| Strategia | Opis |
|---|---|
| `st.integers(min_value=0, max_value=100)` | liczby całkowite |
| `st.floats(min_value=0.0, allow_nan=False)` | liczby zmiennoprzecinkowe |
| `st.text(min_size=1, max_size=50)` | ciągi tekstowe |
| `st.lists(st.integers(), min_size=1)` | listy |
| `st.booleans()` | wartości logiczne |
| `st.sampled_from(['a', 'b', 'c'])` | wybór z listy |
| `st.one_of(st.integers(), st.text())` | jeden z wielu typów |

Dekorator `@settings` kontroluje zachowanie hypothesis:

```python
from hypothesis import given, settings

@settings(max_examples=500, deadline=None)
@given(st.integers())
def test_something(x): ...
```

Shrinking - gdy hypothesis znajdzie błąd, automatycznie minimalizuje
kontrprzykład. Np. dla błędu przy liście `[3, -7, 15]` zredukuje
do `[0]` lub innej prostszej formy pokazującej problem.

In [ ]:
# Przykład: różne strategie
_run('''
from hypothesis import given, settings, assume
import hypothesis.strategies as st

# st.integers z zakresem
@given(st.integers(min_value=0, max_value=1000))
def test_square_nonnegative(n):
    assert n ** 2 >= 0

# st.text - własność strip
@given(st.text())
def test_strip_idempotent(s):
    assert s.strip() == s.strip().strip()

# st.lists + st.integers - własność min/max
@given(st.lists(st.integers(), min_size=1))
def test_min_in_list(lst):
    assert min(lst) in lst

# st.floats z assume - unikamy NaN i dzielenia przez 0
@given(st.floats(allow_nan=False, allow_infinity=False),
       st.floats(allow_nan=False, allow_infinity=False))
def test_addition_commutative(a, b):
    assume(abs(a) < 1e10 and abs(b) < 1e10)
    assert abs((a + b) - (b + a)) < 1e-9
''')

# Shrinking demo - celowy błąd by zobaczyć minimalny kontrprzykład
_run('''
from hypothesis import given, settings
import hypothesis.strategies as st

@settings(max_examples=200)
@given(st.lists(st.integers()))
def test_no_list_longer_than_5(lst):
    # celowo błędna własność - hypothesis znajdzie min. kontrprzykład
    assert len(lst) <= 5
''', flags=["-v", "--tb=short", "-p", "no:cacheprovider"])

---

### 🐍 Ćwiczenia — strategie i @given

3. *(Trudniejsze)* Sprawdź własność przemienności
   `add(a, b) == add(b, a)`
4. Użyj `st.integers(min_value=1)` i `assume(b != 0)` dla
   `divide(a, b)`
5. Testuj `encode/decode` - round-trip dla `base64`
7. Sprawdź własność idempotentności `strip(strip(s)) == strip(s)`
9. Użyj `@settings(max_examples=500)` i obejrzyj shrinking przy
   błędzie

In [ ]:
# Ćwiczenie 3: przemienność add(a, b) == add(b, a) *(Trudniejsze)*
# Napisz property testy sprawdzające własności arytmetyczne:
#   a) przemienność: add(a, b) == add(b, a)
#   b) łączność: add(add(a, b), c) == add(a, add(b, c))
#   c) element neutralny: add(a, 0) == a

_run('''
from hypothesis import given
import hypothesis.strategies as st

def add(a, b):
    return a + b

ints = st.integers(min_value=-10**6, max_value=10**6)

@given(ints, ints)
def test_add_commutative(a, b):
    ...

@given(ints, ints, ints)
def test_add_associative(a, b, c):
    ...

@given(ints)
def test_add_neutral_element(a):
    ...
''')

In [ ]:
# Ćwiczenie 4: divide(a, b) z assume(b != 0)
# Własność: a / b * b powinno być w przybliżeniu równe a.
# Użyj assume(), żeby odfiltrować b == 0.

_run('''
from hypothesis import given, assume
import hypothesis.strategies as st

def divide(a, b):
    return a / b

@given(st.integers(min_value=-1000, max_value=1000),
       st.integers(min_value=-1000, max_value=1000))
def test_divide_inverse(a, b):
    assume(...)
    result = divide(a, b)
    assert abs(result * b - a) < 1e-9
''')

In [ ]:
# Ćwiczenie 5: base64 round-trip
# Własność: decode(encode(data)) == data dla dowolnych bajtów.
# Użyj st.binary() jako strategii dla danych binarnych.

_run('''
from hypothesis import given
import hypothesis.strategies as st
import base64

@given(st.binary())
def test_base64_roundtrip(data):
    encoded = base64.b64encode(data)
    decoded = ...
    assert decoded == data

@given(st.text())
def test_utf8_roundtrip(text):
    encoded = text.encode("utf-8")
    decoded = ...
    assert decoded == text
''')

In [ ]:
# Ćwiczenie 7: idempotentność strip
# Własność idempotentności: f(f(x)) == f(x)
# Sprawdź dla: strip(), lower(), sorted()

_run('''
from hypothesis import given
import hypothesis.strategies as st

@given(st.text())
def test_strip_idempotent(s):
    assert s.strip() == ...

@given(st.text())
def test_lower_idempotent(s):
    assert s.lower() == ...

@given(st.lists(st.integers()))
def test_sorted_idempotent(lst):
    assert sorted(lst) == ...
''')

In [ ]:
# Ćwiczenie 9: @settings(max_examples=500) + shrinking
# Użyj @settings(max_examples=500) dla dokładniejszego testu.
# Napisz celowo błędną funkcję is_palindrome i obserwuj
# jak hypothesis minimalizuje kontrprzykład.

_run('''
from hypothesis import given, settings
import hypothesis.strategies as st

def is_palindrome(s):
    # celowy błąd: nie obsługuje dużych liter
    return s == s[::-1]

# Test poprawny - przejdzie
@settings(max_examples=500)
@given(st.text(alphabet=st.characters(whitelist_categories=("Ll",))))
def test_reverse_of_palindrome_is_palindrome(s):
    # palindrom odwrócony to nadal palindrom
    combined = s + s[::-1]
    assert is_palindrome(combined)
''', flags=["-v", "--tb=short", "-p", "no:cacheprovider"])

## 3. 🔹 `assume`, `composite` i strategie złożone

`assume(condition)` odrzuca przypadek testowy gdy warunek jest fałszywy
- hypothesis szuka następnego spełniającego przypadku:

```python
from hypothesis import given, assume

@given(st.integers(), st.integers())
def test_modulo(a, b):
    assume(b != 0)  # odrzuć b=0
    assert 0 <= a % b < abs(b)
```

`st.composite` pozwala budować złożone strategie jako funkcje:

```python
from hypothesis.strategies import composite

@composite
def user_strategy(draw):
    name = draw(st.text(min_size=1, max_size=50))
    age = draw(st.integers(min_value=18, max_value=80))
    return {'name': name, 'age': age}

@given(user_strategy())
def test_user(user): ...
```

**Stateful testing** (`RuleBasedStateMachine`) modeluje sekwencje
operacji na obiekcie i sprawdza niezmienniki po każdej operacji:

```python
from hypothesis.stateful import RuleBasedStateMachine, rule, invariant

class StackMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.stack = []

    @rule(value=st.integers())
    def push(self, value):
        self.stack.append(value)

    @invariant()
    def stack_len_nonnegative(self):
        assert len(self.stack) >= 0
```

In [ ]:
# Przykład: st.composite - strategia dla prawidłowego hasła
_run('''
from hypothesis import given
from hypothesis.strategies import composite
import hypothesis.strategies as st

@composite
def password_strategy(draw):
    length = draw(st.integers(min_value=8, max_value=20))
    chars = draw(st.text(
        alphabet="abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
        min_size=length, max_size=length
    ))
    return chars

def validate_password(pwd):
    return len(pwd) >= 8

@given(password_strategy())
def test_generated_passwords_valid(pwd):
    assert validate_password(pwd)
    assert len(pwd) >= 8
''')

# Przykład: stateful testing dla Stack
_run('''
from hypothesis.stateful import RuleBasedStateMachine, rule, invariant
import hypothesis.strategies as st

class Stack:
    def __init__(self):
        self._data = []

    def push(self, item):
        self._data.append(item)

    def pop(self):
        if not self._data:
            raise IndexError("pop from empty stack")
        return self._data.pop()

    def peek(self):
        if not self._data:
            raise IndexError("peek at empty stack")
        return self._data[-1]

    def size(self):
        return len(self._data)

class StackMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.stack = Stack()
        self.model = []

    @rule(value=st.integers())
    def push(self, value):
        self.stack.push(value)
        self.model.append(value)

    @rule()
    def pop_if_nonempty(self):
        if self.model:
            result = self.stack.pop()
            expected = self.model.pop()
            assert result == expected

    @invariant()
    def size_matches_model(self):
        assert self.stack.size() == len(self.model)

TestStack = StackMachine.TestCase
''')

---

### 🐍 Ćwiczenia — strategie złożone i stateful

6. *(Trudniejsze)* Własna strategia `st.composite` dla klasy `User`
10. *(Trudniejsze)* Sprawdź własności klasy `BankAccount` przez wiele
    operacji
11. Przetestuj parser JSON - każdy poprawny JSON powinien się sparsować
12. *(Trudniejsze)* Stateful testing - `@rule`, `@invariant` dla `Stack`

In [ ]:
# Ćwiczenie 6: st.composite dla klasy User *(Trudniejsze)*
# Napisz strategię user_strategy() generującą prawidłowych użytkowników:
#   - name: niepusty tekst, max 50 znaków
#   - email: 'user' + cyfry + '@example.com'
#   - age: liczba całkowita 18-100
# Użyj strategii w teście rejestracji.

_run('''
from hypothesis import given
from hypothesis.strategies import composite
import hypothesis.strategies as st

@composite
def user_strategy(draw):
    name = draw(st.text(min_size=1, max_size=50))
    uid = draw(st.integers(min_value=1000, max_value=9999))
    age = draw(st.integers(min_value=..., max_value=...))
    return {
        "name": name,
        "email": f"user{uid}@example.com",
        "age": age,
    }

def is_valid_user(user):
    return (
        len(user["name"]) > 0
        and "@" in user["email"]
        and user["age"] >= 18
    )

@given(user_strategy())
def test_generated_users_are_valid(user):
    assert is_valid_user(user)
''')

In [ ]:
# Ćwiczenie 10: BankAccount property tests *(Trudniejsze)*
# Sprawdź następujące własności klasy BankAccount:
#   a) po deposit(n) saldo wzrasta o n
#   b) po withdraw(n) saldo maleje o n (gdy wystarczające środki)
#   c) saldo nigdy nie jest ujemne
#   d) deposit(a) + deposit(b) == deposit(a+b) (addytywność)

_run('''
from hypothesis import given, assume
import hypothesis.strategies as st

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount

pos_int = st.integers(min_value=1, max_value=10_000)

@given(pos_int, pos_int)
def test_deposit_increases_balance(initial, amount):
    acc = BankAccount(initial)
    acc.deposit(amount)
    assert acc.balance == ...

@given(pos_int, pos_int)
def test_withdraw_decreases_balance(initial, amount):
    assume(amount <= initial)
    acc = BankAccount(initial)
    acc.withdraw(amount)
    assert acc.balance == ...

@given(pos_int, pos_int)
def test_balance_never_negative(initial, amount):
    assume(amount <= initial)
    acc = BankAccount(initial)
    acc.withdraw(amount)
    assert acc.balance >= 0

@given(pos_int, pos_int, pos_int)
def test_deposit_additive(initial, a, b):
    acc1 = BankAccount(initial)
    acc1.deposit(a)
    acc1.deposit(b)
    acc2 = BankAccount(initial)
    acc2.deposit(a + b)
    assert acc1.balance == ...
''')

In [ ]:
# Ćwiczenie 11: parser JSON round-trip
# Własność: json.dumps(data) -> json.loads -> ten sam wynik.
# Użyj st.recursive() lub st.fixed_dictionaries() do generowania
# danych kompatybilnych z JSON.

_run('''
from hypothesis import given
import hypothesis.strategies as st
import json

# Strategia dla prostych wartości JSON
json_primitive = st.one_of(
    st.none(),
    st.booleans(),
    st.integers(),
    st.floats(allow_nan=False, allow_infinity=False),
    st.text(),
)

@given(json_primitive)
def test_json_primitive_roundtrip(value):
    serialized = json.dumps(value)
    deserialized = json.loads(serialized)
    assert deserialized == value

@given(st.lists(st.integers()))
def test_json_list_roundtrip(lst):
    assert json.loads(json.dumps(lst)) == ...

@given(st.dictionaries(st.text(min_size=1), st.integers()))
def test_json_dict_roundtrip(d):
    assert json.loads(json.dumps(d)) == ...
''')

In [ ]:
# Ćwiczenie 12: Stateful testing dla Stack *(Trudniejsze)*
# Zaimplementuj StackMachine testującą klasę Stack.
# Reguły: push(value), pop_if_nonempty, peek_if_nonempty
# Niezmienniki:
#   - size() >= 0 zawsze
#   - push + pop zwraca tę samą wartość
#   - peek nie zmienia rozmiaru

_run('''
from hypothesis.stateful import RuleBasedStateMachine, rule, invariant
import hypothesis.strategies as st

class Stack:
    def __init__(self):
        self._data = []

    def push(self, item):
        self._data.append(item)

    def pop(self):
        return self._data.pop()

    def peek(self):
        return self._data[-1]

    def size(self):
        return len(self._data)

    def is_empty(self):
        return len(self._data) == 0

class StackMachine(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.stack = Stack()
        self.model = []

    @rule(value=st.integers())
    def push(self, value):
        self.stack.push(value)
        self.model.append(value)

    @rule()
    def pop_if_nonempty(self):
        if self.model:
            result = self.stack.pop()
            expected = self.model.pop()
            assert result == expected

    @rule()
    def peek_does_not_change_size(self):
        if self.model:
            size_before = self.stack.size()
            self.stack.peek()
            assert self.stack.size() == size_before

    @invariant()
    def size_nonnegative(self):
        assert self.stack.size() >= 0

    @invariant()
    def size_matches_model(self):
        assert self.stack.size() == len(self.model)

TestStack = StackMachine.TestCase
''')